---

Image datasets and measurement

---

In [ ]:
# autoload
%load_ext autoreload
%autoreload 2

# Load PGL libraries and start a PGL window
from pgl import pgl
from pgl.pglImage import pglImageDatabase, pglImageDatabaseNSD, pglImage
from pgl.pglMessages import pglMessages
from pgl.pglExperiment import pglTask, pglExperiment
from pgl.pglParameter import pglParameter
import numpy as np

pgl = pgl()

# close any existing windows
pgl.cleanUp()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:35:20) [Clang 16.0.6 ]
(pgl:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.5.1
(pgl:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl:checkOS)   Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pglBase) Main library instance created
(

---

Load the image database

---

In [24]:
# load the database of images. Will check in directory for image formats that PIL
# knows about and make a list. This does not load the images, or check to see if they are valid
imdb = pglImageDatabaseNSD(dataPath="ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000")

(pglImageDatabaseNSD->pglImageDatabase:__init__) Found 1000 image files
/Users/justin/Desktop/NSD_shared1000/a_nsd_shared1000_manifest.csv
(pglMessages:warning) ❌ /Users/justin/Desktop/NSD_shared1000/a_nsd_shared1000_manifest.csv does not have a filename column which matches image names: name 'pd' is not defined ❌


---

Print and display images

---

In [16]:
# display a single image
#imdb.images[111].display()

# print image metadata one-by-one, this may take some time because it 
# has to open each file 
#imdb.print() 
imdb.print()

0: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/NSD_shared1000/image_0.png
image_0.png 425x425 RGBA
1: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/NSD_shared1000/image_1.png
image_1.png 425x425 RGBA
2: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/NSD_shared1000/image_2.png
image_2.png 425x425 RGBA
3: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/NSD_shared1000/image_3.png
image_3.png 425x425 RGBA
4: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/NSD_shared1000/image_4.png
image_4.png 425x425 RGBA
5: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Users/justin/Desktop/NSD_shared1000/image_5.png
image_5.png 425x425 RGBA
6: (pglImageFile:_loadMetadata) Loading image metadata: ssh://justin@lagavulin//Us

KeyboardInterrupt: 

---

Display image dataset in a dialog

---

In [18]:
pgl.traitsDialog(imdb)

❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌❌
(pglSerialize:decodeObject) (pglSerialize) Could not restore object of class 'pglImageDatabaseNSD' from '/var/folders/n2/4bqdpl1s6wd9x4l9ll4p03_m0000gq/T/tmpk1blhwaw/in.json'.
Known classes: ['pglAction', 'pglActionLoadSession', 'pglActionLoadSessionSettings', 'pglActionRecreateExperimentDataFromTasks', 'pglActionRecreateExperimentDataFromTasksChooseRun', 'pglActionRecreateExperimentDataFromTasksChooseTaskName', 'pglActionRecreateExperimentDataFromTasksSettings', 'pglActionSave', 'pglAnalogTraceData', 'pglBarTask', 'pglChoose', 'pglChooseExperiment', 'pglChooseLevel', 'pglChooseRun', 'pglChooseSession', 'pglChooseSubject', 'pglDisplayLuminanceCalibrationData', 'pglDisplayModeSettings', 'pglDisplaySettings', 'pglDisplaySettingsList', 'pglDisplaySettingsWindowed', 'pglDisplayTemporalCalibrationData', 'pglEvent', 'pglEventBlink', 'pglEventEyeTrackerTrial', 'pglEventKeyboard', 'pglEventResponsePixx', 'pglEvent

In [ ]:
img=imdb.getImage(0)

---

Make a task to display images

---

In [ ]:
class pglImageTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Image Task"
        
        # set seglens, 
        # 1st segment is image display
        # 2nd segment is blank
        self.settings.seglen = [0.5, 0.5]

        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            'imagesDirectory': "ssh://justin@lagavulin/Users/justin/Desktop/NSD_shared1000",
            'imdb': None,
            'nImages': 50,
            'imageSize': 18
        }        
        p = self.settings.fixedParameters
        
        # initialize image database
        imdb = pglImageDatabase(p['imagesDirectory'])
        if imdb.nImages==0:
            pglMessages.warning(f"No images found in {p['imageDirectory']}")
        p['imdb'] = imdb
        
        # preload images
        for iImage in range(p['nImages']):
            imdb.preloadImage(iImage)
            
        # add parameter for image number
        imageNum = pglParameter('imageNum',np.arange(p['nImages']))
        self.addParameter(imageNum)
        
        # set current image
        self.state.currentImage = None

    ########################
    def startSegment(self, startTime):
        '''
        Start a segment
        '''
        super().startSegment(startTime)
    
        # load the image
        if self.state.currentSegment == 0: 
            # get image database
            imdb = self.settings.fixedParameters['imdb']
            # get the current image number
            imageNum = self.currentParams['imageNum']
            # get the image data
            img = imdb.getImage(imageNum)
            img.convert("RGB")
            print(f"img: {img}")
            # turn into a pglImage
            self.state.currentImage = self.pgl.imageCreate(np.array(img))
    
    ########################
    # updateScren
    ########################
    def updateScreen(self):
        '''
        update the screen
        '''
        if self.state.currentSegment == 1: 
            if self.state.currentImage:
                self.state.currentImage.display(height=self.settings.fixedParameters['imageSize'])
        
        # Draw ABC fixation cross from Thaler, Schütz, Goodale & Gegenfurtner (2013) Vision Research 76:31-42
        pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
        pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
        pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
        pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

        


---

Setup experiment

---

In [ ]:
pgl.cleanUp()
#e = pglExperiment(pgl,settingsName='Cinema',experimentName='imageTask')
e = pglExperiment(pgl,experimentName='imageTask')

imageTask = pglImageTask(pgl)
e.addTask(imageTask)

---

run experiment

---

In [ ]:
e.initScreen()
e.run()

In [ ]:
e.tasks[0].settings.fixedParameters

In [ ]:
pgl.open(0)

In [ ]:
pgl.arc(0,0,0,0.3,stopAngle=2*np.pi,borderSize=0,color=0)
pgl.rect(-0.3,0,width=0.6,height=0.15,color=1,hAlign='left',vAlign='center')
pgl.rect(0,-0.3,width=0.15,height=0.6,color=1,hAlign='center',vAlign='top')
pgl.arc(0,0,0,0.1,stopAngle=2*np.pi,borderSize=0,color=0)

pgl.flush()